In [ ]:
import pandas as pd
from datetime import datetime
from vnstock.api.quote import Quote
import time

def crawl_and_process_data(symbol, start_date, end_date, resolution='1m', source='VCI'):
    """
    Hàm cào dữ liệu, lọc chính xác theo ngày và lưu file tự động.
    """
    print(f"🚀 Bắt đầu lấy dữ liệu mã: {symbol} | Phân giải: {resolution} | Nguồn: {source}")
    print(f"📅 Yêu cầu lọc từ: {start_date} đến {end_date}")
    
    try:
        # 1. Khởi tạo và gọi API
        quote_api = Quote(symbol=symbol, source=source)
        df = quote_api.history(start=start_date, end=end_date, resolution=resolution)
        
        # 2. Kiểm tra dữ liệu rỗng từ API
        if df is None or df.empty:
            print(f"⚠️ Cảnh báo: API {source} không trả về dữ liệu cho khoảng thời gian này.")
            return None
            
        print(f"📥 Đã tải về chunk dữ liệu thô: {len(df)} dòng.")
        
        # ==========================================
        # BƯỚC XỬ LÝ DỮ LIỆU (DATA PROCESSING)
        # ==========================================
        
        # 3. Ép kiểu cột 'time' về chuẩn Datetime (Bắt buộc)
        df['time'] = pd.to_datetime(df['time'])
        
        # 4. Ép kiểu input start_date và end_date để so sánh cho chuẩn
        # Lưu ý: end_date thêm 23:59:59 để bao gồm trọn vẹn ngày cuối cùng
        start_dt = pd.to_datetime(start_date)
        end_dt = pd.to_datetime(end_date).replace(hour=23, minute=59, second=59)
        
        # 5. Dùng Mask để lọc chính xác dữ liệu nằm trong khoảng yêu cầu
        mask = (df['time'] >= start_dt) & (df['time'] <= end_dt)
        df_filtered = df.loc[mask].copy()
        
        # Kiểm tra xem sau khi lọc còn dữ liệu không
        if df_filtered.empty:
            print("⚠️ Cảnh báo: Sau khi lọc theo ngày yêu cầu, không còn dòng dữ liệu nào.")
            print(f"Khoảng dữ liệu thô thực tế API trả về là từ: {df['time'].iloc[0]} đến {df['time'].iloc[-1]}")
            return None
            
        # Reset lại index sau khi lọc
        df_filtered.reset_index(drop=True, inplace=True)
        
        # ==========================================
        # BƯỚC ĐẶT TÊN VÀ XUẤT FILE
        # ==========================================
        
        # 6. Trích xuất mốc thời gian THỰC TẾ sau khi lọc để đặt tên file
        actual_start = df_filtered['time'].iloc[0]
        actual_end = df_filtered['time'].iloc[-1]
        
        # Định dạng ngày thành chuỗi DDMMYY (Ví dụ: 040526)
        start_str = actual_start.strftime('%d%m%y')
        end_str = actual_end.strftime('%d%m%y')
        
        # 7. Ghép tên file theo đúng cú pháp yêu cầu
        csv_filename = f"/Dataset/{symbol}_{resolution}_{start_str}{end_str}.csv"
        
        # Xuất file CSV
        df_filtered.to_csv(csv_filename, index=False, encoding='utf-8')
        
        print(f"✂️  Đã lọc dữ liệu thành công! Còn lại: {len(df_filtered)} dòng.")
        print(f"🕒 Dữ liệu thực tế: Từ {actual_start} đến {actual_end}")
        print(f"💾 Đã lưu file tại: {csv_filename}\n")
        
        return df_filtered

    except Exception as e:
        print(f"❌ Xảy ra lỗi trong quá trình thực thi: {e}")
        return None

📦 **Vnstock 4.0.4 is available**
Current: 4.0.3 (Python 3.11 (venv))
Update: `D:\classes\HPC\Do_an\.venv_3119\Scripts\python.exe -m pip install vnstock --upgrade`
Release: https://vnstocks.com/docs/tai-lieu/lich-su-phien-ban

In [3]:
df_result = crawl_and_process_data(
    symbol="FPT", 
    start_date="2025-01-01", 
    end_date="2026-05-17"
)

🚀 Bắt đầu lấy dữ liệu mã: FPT | Phân giải: 1m | Nguồn: VCI
📅 Yêu cầu lọc từ: 2025-01-01 đến 2026-05-17
📥 Đã tải về chunk dữ liệu thô: 91546 dòng.
✂️  Đã lọc dữ liệu thành công! Còn lại: 75856 dòng.
🕒 Dữ liệu thực tế: Từ 2025-01-02 09:15:00 đến 2026-05-15 14:45:00
💾 Đã lưu file tại: FPT_1m_020125150526.csv

